In [3]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle, RegularPolygon, Polygon
from matplotlib.backends.backend_pdf import PdfPages
import random
import os
import torch

from sparse_generalization.scripts.shape_dataset import is_same_row, generate_grid_row, generate_grid_adjacent, generate_grid_row_col, generate_grid_same_row
from sparse_generalization.data.shapes.constants import SHAPE_COLORS, SHAPES_TO_IDX, SHAPES, IDX_TO_SHAPES, SHAPE_MAP
from sparse_generalization.utils.util_funcs import positionalencoding2d
from sparse_generalization.models.spartan import SPARTAN

%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
# Shapes and mapping
shapes = ["circle","square","triangle","diamond","pentagon","heart","star","moon","plus"]
shape_to_idx = {s:i for i,s in enumerate(shapes)}
shape_colors = {
    "circle": "red",
    "square": "blue",
    "triangle": "green",
    "diamond": "purple",
    "pentagon": "orange",
    "heart": "pink",
    "star": "gold",
    "moon": "black",
    "plus": "cyan"
}

def draw_shape(ax, shape, x, y, size=0.35):
    color = shape_colors[shape]
    if shape == "circle":
        ax.add_patch(Circle((x, y), size, color=color))
    elif shape == "square":
        ax.add_patch(Rectangle((x-size, y-size), 2*size, 2*size, color=color))
    elif shape == "triangle":
        ax.add_patch(RegularPolygon((x,y), 3, radius=size, color=color))
    elif shape == "diamond":
        verts = [(x,y+size),(x+size,y),(x,y-size),(x-size,y)]
        ax.add_patch(Polygon(verts, color=color))
    elif shape == "pentagon":
        ax.add_patch(RegularPolygon((x,y),5,radius=size,color=color))
    elif shape == "heart":
        t = np.linspace(0,2*np.pi,100)
        hx = size*16*np.sin(t)**3
        hy = size*(13*np.cos(t)-5*np.cos(2*t)-2*np.cos(3*t)-np.cos(4*t))
        hx = hx/18 + x
        hy = hy/18 + y
        ax.add_patch(Polygon(np.column_stack([hx,hy]), color=color))
    elif shape == "star":
        pts=[]
        for i in range(10):
            angle=i*np.pi/5
            r=size if i%2==0 else size*0.4
            pts.append((x+r*np.cos(angle), y+r*np.sin(angle)))
        ax.add_patch(Polygon(pts,color=color))
    elif shape == "moon":
        moon = Circle((x,y), size, color=color)
        ax.add_patch(moon)
        cover = Circle((x+size*0.4,y+size*0.1), size, color="white")
        ax.add_patch(cover)
    elif shape == "plus":
        t = size*0.6
        ax.add_patch(Rectangle((x-t/2,y-size), t,2*size, color=color))
        ax.add_patch(Rectangle((x-size,y-t/2), 2*size,t, color=color))

# Generate random grid
def generate_random_grid():
    rows, cols = 3,3
    grid = np.empty((rows,cols),dtype=object)
    remaining = shapes.copy()
    random.shuffle(remaining)
    idx = 0
    for r in range(rows):
        for c in range(cols):
            grid[r,c] = remaining[idx]
            idx += 1
    return grid

# Horizontal adjacency check
def is_horizontal_adjacent(grid, shape1, shape2):
    for r in range(3):
        for c in range(2):
            if grid[r,c] == shape1 and grid[r,c+1] == shape2:
                return True
            if grid[r,c] == shape2 and grid[r,c+1] == shape1:
                return True
    return False


In [8]:
# Dataset generation, equal amount of each label.
num_images = 250
half_images = num_images // 2
save_dir = "../data/shapes/"
mode = 'train' #other options: test_a, test_b
func = 'row_col'
size = 5
os.makedirs(save_dir, exist_ok=True)

all_tensors = []
labels = []
label_counts = {"a":0, "b":0}

i = 0
attempts = 0
max_attempts = num_images * 10

pdf_path = os.path.join(save_dir, f"all_grids_{mode}_{func}.pdf")
pdf_pages = PdfPages(pdf_path)

for label in [True, False]:
    for _ in range(half_images):
        grid = generate_grid_row_col(size=size, label_A=label, mode=mode)

        # Create figure and save to PDF
        fig, ax = plt.subplots(figsize=(5,5))
        for r in range(size):
            for c in range(size):
                x = c + 0.5
                y = size - r - 0.5
                color = SHAPE_COLORS[grid[r,c]]
                obj = SHAPE_MAP[grid[r,c]](x, y, color=color)
                obj.draw(ax)
                
        ax.text(1.5, 0.0, f"Label: {label}", fontsize=16, ha='center', va='bottom', color="black", weight='bold')
        ax.set_xlim(0,size)
        ax.set_ylim(0,size)
        ax.set_aspect("equal")
        ax.axis("off")
        
        pdf_pages.savefig(fig)
        plt.close(fig)
        i += 1

pdf_pages.close()

# Save tensors and labels
# np.save(os.path.join(save_dir,f"grid_tensors_{data_type}.npy"), np.array(all_tensors))
# np.save(os.path.join(save_dir,f"grid_labels_{data_type}.npy"), np.array(labels))